In [2]:
"""
Common Module
"""

import os

from dotenv import load_dotenv

%load_ext autoreload
%autoreload 2

load_dotenv()

BASE_URL = os.getenv("ELASTIC_BASE_URL")
USERNAME = os.getenv("ELASTIC_USER")
PASSWORD = os.getenv("ELASTIC_PASSWORD")
CA_CERT_PATH = os.getenv("ELASTIC_CERT_PATH")

if USERNAME is None or PASSWORD is None:
    raise ValueError("Missing Elastic credentials! Check your .env file.")

## Searching in `search_as_you_type` and its subfields

In [ ]:
"""
Queries the tech_books4 index in Elasticsearch using credentials from a .env file.
"""
import json

import requests

URL = f"{BASE_URL}/tech_books4/_search"

PAYLOAD = {
    "query": {
        "multi_match": {
            "query": "in",
            "type": "bool_prefix",
            "fields": ["title", "title._2gram", "title._3gram"],
        }
    }
}

response = requests.post(
    url=URL, auth=(USERNAME, PASSWORD), verify=CA_CERT_PATH, json=PAYLOAD
)

print(json.dumps(response.json(), indent=2))

{
  "took": 20,
  "timed_out": false,
  "_shards": {
    "total": 1,
    "successful": 1,
    "skipped": 0,
    "failed": 0
  },
  "hits": {
    "total": {
      "value": 2,
      "relation": "eq"
    },
    "max_score": 1.0,
    "hits": [
      {
        "_index": "tech_books4",
        "_id": "1",
        "_score": 1.0,
        "_source": {
          "title": "Elasticsearch in Action"
        }
      },
      {
        "_index": "tech_books4",
        "_id": "3",
        "_score": 1.0,
        "_source": {
          "title": "Elastic Stack in Action"
        }
      }
    ]
  }
}


## Scenario 1: The Raw Prefix Match

In [ ]:
"""
What it tests: How _index_prefix catches unfinished words right at the beginning.
"""
import json

import requests

URL = f"{BASE_URL}/tech_books4/_search"

PAYLOAD = {
    "query": {
        "multi_match": {
            "query": "elast",
            "type": "bool_prefix",
            "fields": ["title", "title._2gram", "title._3gram"],
        }
    }
}

response = requests.post(
    url=URL, auth=(USERNAME, PASSWORD), verify=CA_CERT_PATH, json=PAYLOAD
)

print(json.dumps(response.json(), indent=2))

{
  "took": 5,
  "timed_out": false,
  "_shards": {
    "total": 1,
    "successful": 1,
    "skipped": 0,
    "failed": 0
  },
  "hits": {
    "total": {
      "value": 3,
      "relation": "eq"
    },
    "max_score": 1.0,
    "hits": [
      {
        "_index": "tech_books4",
        "_id": "1",
        "_score": 1.0,
        "_source": {
          "title": "Elasticsearch in Action"
        }
      },
      {
        "_index": "tech_books4",
        "_id": "2",
        "_score": 1.0,
        "_source": {
          "title": "Elasticsearch for Java Developers"
        }
      },
      {
        "_index": "tech_books4",
        "_id": "3",
        "_score": 1.0,
        "_source": {
          "title": "Elastic Stack in Action"
        }
      }
    ]
  }
}


## Scenario2: The Space Trigger (The Eliminator)

In [4]:
"""
What it tests: How the engines uses spaces to declare a word "completed".
"""

import json

import requests

URL = f"{BASE_URL}/tech_books4/_search"

PAYLOAD = {
    "query": {
        "multi_match": {
            "query": "elastic ",
            "type": "bool_prefix",
            "fields": ["title", "title._2gram", "title._3gram"],
        }
    }
}

response = requests.post(
    url=URL, auth=(USERNAME, PASSWORD), verify=CA_CERT_PATH, json=PAYLOAD
)

print(json.dumps(response.json(), indent=2))

{
  "took": 81,
  "timed_out": false,
  "_shards": {
    "total": 1,
    "successful": 1,
    "skipped": 0,
    "failed": 0
  },
  "hits": {
    "total": {
      "value": 1,
      "relation": "eq"
    },
    "max_score": 0.94566,
    "hits": [
      {
        "_index": "tech_books4",
        "_id": "3",
        "_score": 0.94566,
        "_source": {
          "title": "Elastic Stack in Action"
        }
      }
    ]
  }
}
